# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/jamieleeuw/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

Unit of analysis: one row = one content item, for one client, on one calendar day
(report_date + client_hash_id + content_hash_id). This is the daily fact grain from
fact_content_daily_performance — one row per page-day, not one row per page overall.

Time window: month=2026-03 (a mid-panel month), per the lane guide's warning that
the _sample table (the final month) should be treated as a sealed test month, not
used to develop label logic.

In [1]:
import duckdb
import os
from dotenv import load_dotenv, find_dotenv

load_dotenv(find_dotenv())

con = duckdb.connect()
con.execute("INSTALL httpfs;")
con.execute("LOAD httpfs;")

con.execute(f"""
    CREATE SECRET hf_token (
        TYPE HUGGINGFACE,
        TOKEN '{os.environ["HF_TOKEN"]}'
    );
""")

result = con.execute("""
    SELECT COUNT(*) AS n_clients
    FROM 'hf://datasets/FlyRank/internship-warehouse/dim_clients.parquet'
""").df()
print(result)

result = con.execute("""
    SELECT *
    FROM 'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet'
    LIMIT 5
""").df()
print(result.columns.tolist())
result

   n_clients
0        104
['report_date', 'client_hash_id', 'content_hash_id', 'client_has_gsc', 'client_has_ga4', 'gsc_data_available', 'ga4_data_available', 'gsc_impressions', 'gsc_clicks', 'gsc_sum_position', 'gsc_avg_position', 'ga4_pageviews', 'ga4_sessions', 'ga4_users', 'ga4_engaged_sessions', 'ga4_total_engagement_sec', 'sessions_organic', 'sessions_direct', 'sessions_referral', 'sessions_social', 'sessions_paid', 'sessions_ai', 'ai_chatgpt', 'ai_perplexity', 'ai_gemini', 'ai_copilot', 'ai_claude', 'ai_meta', 'ai_other', 'scroll_events', 'month']


,report_date,client_hash_id,content_hash_id,client_has_gsc,client_has_ga4,gsc_data_available,ga4_data_available,gsc_impressions,gsc_clicks,gsc_sum_position,...,sessions_ai,ai_chatgpt,ai_perplexity,ai_gemini,ai_copilot,ai_claude,ai_meta,ai_other,scroll_events,month
0,2026-03-01,client_73cda7b4e4f265ea,content_b7e512995f79d5a6,True,False,True,<NA>,20,0,67,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
1,2026-03-01,client_73cda7b4e4f265ea,content_05597932fe4da067,True,False,True,<NA>,1,0,0,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
2,2026-03-01,client_73cda7b4e4f265ea,content_7a105f548d9c6916,True,False,True,<NA>,125,1,616,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
3,2026-03-01,client_73cda7b4e4f265ea,content_905aa32a0230694e,True,False,True,<NA>,7,0,28,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
4,2026-03-01,client_73cda7b4e4f265ea,content_a3ea9792f793ec72,True,False,True,<NA>,11,0,25,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

Feature (known at the decision moment, safe as model inputs):
gsc_impressions, gsc_clicks, gsc_avg_position, gsc_sum_position, ga4_sessions,
ga4_engaged_sessions, ga4_pageviews, ga4_users, ga4_total_engagement_sec,
sessions_organic/direct/referral/social/paid, scroll_events — all observed
same-day search/engagement signals.

Label / proxy: not shipped directly — this is a daily fact table, not a labeled
one. I define a proxy myself: "low-CTR visible page" (decent impressions and
position, but weak CTR), mirroring the starter baseline's low_ctr_visible_page
reason code.

Context (for joining/filtering, not model inputs): report_date, client_hash_id,
content_hash_id, client_has_gsc, client_has_ga4, gsc_data_available,
ga4_data_available, month.

Excluded: ai_chatgpt, ai_perplexity, ai_gemini, ai_copilot, ai_claude, ai_meta,
ai_other, sessions_ai — too sparse in a single month for reliable features, and
belong to the separate AI Referral Opportunity lane, not this one.

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

Five features, each knowable at the decision moment (same day the row describes):
1. gsc_ctr — clicks/impressions logged by Search Console same-day.
2. log_impressions — same-day impressions, log-transformed to reduce skew.
3. has_sessions — GA4 sessions recorded same-day.
4. has_scroll_events — scroll events logged same-day.
5. avg_position — Search Console's average position for that day's impressions.

Leakage trap: I defined a proxy label ("low-CTR visible page": impressions >= 50,
position 1-20, CTR < 2%) and deliberately included gsc_ctr — the exact field the
label is built from — as a model feature. AUC with gsc_ctr included: 1.000, a
dead giveaway of leakage, since the model just re-detected its own label rule.

After removing gsc_ctr, the AUC only dropped to 0.997 — still suspiciously high.
The reason: my label definition also uses avg_position and gsc_impressions as
direct thresholds, and those remained in the "honest" feature set too. So even
without gsc_ctr, the model could still nearly reconstruct the label from the same
cutoffs used to define it. This is a second, subtler form of the same leakage
lesson from notebook 2's trend_pct: leakage isn't only "the exact column" — any
feature that's structurally part of the label's definition leaks too. A properly
honest evaluation here would need a label defined from a genuinely separate
signal (e.g. a future-window outcome) rather than same-day thresholds on the same
fields used as features.

In [3]:
grain_check = con.execute("""
    SELECT COUNT(*) AS total_rows, COUNT(*) - COUNT(DISTINCT (report_date, client_hash_id, content_hash_id)) AS duplicate_rows
    FROM 'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet'
""").df()
print("Grain check:\n", grain_check)

span_check = con.execute("""
    SELECT COUNT(*) AS row_count, MIN(report_date) AS first_date, MAX(report_date) AS last_date
    FROM 'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet'
""").df()
print("\nRow count / date span:\n", span_check)

availability_check = con.execute("""
    SELECT COUNT(*) AS total_rows,
      SUM(CASE WHEN gsc_data_available IS TRUE THEN 1 ELSE 0 END) AS gsc_available_rows,
      SUM(CASE WHEN ga4_data_available IS TRUE THEN 1 ELSE 0 END) AS ga4_available_rows
    FROM 'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet'
""").df()
print("\nAvailability check:\n", availability_check)
import numpy as np
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import roc_auc_score

slice_df = con.execute("""
    SELECT report_date, client_hash_id, content_hash_id, gsc_impressions, gsc_clicks,
           gsc_avg_position, ga4_sessions, scroll_events
    FROM 'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet'
    WHERE gsc_data_available IS TRUE AND gsc_impressions > 0
""").df()
print(f"Working slice rows: {len(slice_df):,}")

slice_df["gsc_ctr"] = slice_df["gsc_clicks"] / slice_df["gsc_impressions"]
slice_df["log_impressions"] = np.log1p(slice_df["gsc_impressions"])
slice_df["has_sessions"] = (slice_df["ga4_sessions"].fillna(0) > 0).astype(int)
slice_df["has_scroll_events"] = (slice_df["scroll_events"].fillna(0) > 0).astype(int)
slice_df["avg_position"] = slice_df["gsc_avg_position"]
features = ["gsc_ctr", "log_impressions", "has_sessions", "has_scroll_events", "avg_position"]
print("\nFeature summary:\n", slice_df[features].describe())

slice_df["low_ctr_label"] = (
    (slice_df["gsc_impressions"] >= 50) &
    (slice_df["avg_position"] > 0) & (slice_df["avg_position"] <= 20) &
    (slice_df["gsc_ctr"] < 0.02)
).astype(int)
print("\nLabel rate:", slice_df["low_ctr_label"].mean())

X_leaky = slice_df[features].fillna(0)
y = slice_df["low_ctr_label"]
leaky_model = DecisionTreeClassifier(max_depth=3, random_state=42).fit(X_leaky, y)
leaky_auc = roc_auc_score(y, leaky_model.predict_proba(X_leaky)[:, 1])
print(f"\nLeaky AUC (gsc_ctr included): {leaky_auc:.3f}  <- suspiciously high")

X_honest = slice_df[[f for f in features if f != "gsc_ctr"]].fillna(0)
honest_model = DecisionTreeClassifier(max_depth=3, random_state=42).fit(X_honest, y)
honest_auc = roc_auc_score(y, honest_model.predict_proba(X_honest)[:, 1])
print(f"Honest AUC (gsc_ctr removed): {honest_auc:.3f}")

Grain check:
    total_rows  duplicate_rows
0     9841378               0

Row count / date span:
    row_count first_date  last_date
0    9841378 2026-03-01 2026-03-31

Availability check:
    total_rows  gsc_available_rows  ga4_available_rows
0     9841378           3611061.0            413966.0
Working slice rows: 3,611,061

Feature summary:
             gsc_ctr  log_impressions  has_sessions  has_scroll_events  \
count  3.611061e+06     3.611061e+06  3.611061e+06       3.611061e+06   
mean   3.080748e-03     2.966355e+00  9.999693e-02       2.995823e-02   
std    3.009151e-02     1.616327e+00  2.999959e-01       1.704721e-01   
min    0.000000e+00     6.931472e-01  0.000000e+00       0.000000e+00   
25%    0.000000e+00     1.609438e+00  0.000000e+00       0.000000e+00   
50%    0.000000e+00     2.833213e+00  0.000000e+00       0.000000e+00   
75%    0.000000e+00     4.143135e+00  0.000000e+00       0.000000e+00   
max    1.000000e+00     1.059876e+01  1.000000e+00       1.000000e+0

## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

Named limitation: this warehouse is an unbalanced panel — clients have different
amounts of history (dim_clients.gsc_data_start / ga4_data_start vary), so a single
month like 2026-03 doesn't represent every client equally. GA4 coverage is also far
thinner than GSC in this month (413,966 GA4-available rows vs. 3,611,061 GSC-available
rows out of 9,841,378 total) — engagement-based features will be missing for most
rows, so any model leaning heavily on GA4 signals effectively only works for a
minority of pages. This data also can never tell us WHY a page's CTR or position
moved (no competitor or SERP context) — only that it did.

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.